# 🏋️ Week 6: Debugging & ML Reasoning Practice

Practice diagnosing model issues and debugging ML systems.

---

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print("Imports ready!")

---
## Exercise 1: Diagnose Overfitting vs Underfitting

Given training/test scores, diagnose the problem and suggest fixes.

**Difficulty:** Medium | **Skill:** ML reasoning

In [ ]:
def diagnose_model(train_score: float, test_score: float, 
                   baseline_score: float = 0.5) -> dict:
    """
    Diagnose model issues based on train/test scores.
    
    Args:
        train_score: Training set accuracy/score
        test_score: Test set accuracy/score
        baseline_score: Random baseline performance
    
    Returns:
        Dict with diagnosis, severity, and recommendations
    """
    # YOUR CODE HERE
    pass


# Test cases
scenarios = [
    (0.99, 0.65),  # High train, low test
    (0.55, 0.52),  # Both low
    (0.92, 0.90),  # Both high, close
    (0.60, 0.75),  # Test > Train (suspicious)
]

for train, test in scenarios:
    result = diagnose_model(train, test)
    print(f"Train: {train:.2f}, Test: {test:.2f} → {result['diagnosis']}")

In [ ]:
# Solution
def diagnose_model_solution(train_score: float, test_score: float, 
                            baseline_score: float = 0.5) -> dict:
    gap = train_score - test_score
    
    # Test score higher than train (data leakage or wrong split)
    if test_score > train_score + 0.05:
        return {
            'diagnosis': 'DATA_LEAKAGE_SUSPECTED',
            'severity': 'critical',
            'recommendations': [
                'Check for data leakage in features',
                'Verify train/test split is correct',
                'Ensure no future information in features'
            ]
        }
    
    # Both scores low (underfitting)
    if train_score < baseline_score + 0.15 and test_score < baseline_score + 0.15:
        return {
            'diagnosis': 'UNDERFITTING',
            'severity': 'high',
            'recommendations': [
                'Use more complex model',
                'Add more features',
                'Reduce regularization',
                'Train longer (if neural net)'
            ]
        }
    
    # High train, low test (overfitting)
    if gap > 0.15:
        severity = 'critical' if gap > 0.25 else 'high'
        return {
            'diagnosis': 'OVERFITTING',
            'severity': severity,
            'recommendations': [
                'Add regularization (L1/L2)',
                'Reduce model complexity',
                'Get more training data',
                'Use dropout (if neural net)',
                'Try cross-validation'
            ]
        }
    
    # Slight overfitting
    if gap > 0.05:
        return {
            'diagnosis': 'SLIGHT_OVERFITTING',
            'severity': 'medium',
            'recommendations': [
                'Consider light regularization',
                'Monitor with cross-validation'
            ]
        }
    
    # Good fit
    return {
        'diagnosis': 'GOOD_FIT',
        'severity': 'none',
        'recommendations': [
            'Model appears well-tuned',
            'Consider ensemble for marginal gains'
        ]
    }

# Test
for train, test in scenarios:
    result = diagnose_model_solution(train, test)
    print(f"Train: {train:.2f}, Test: {test:.2f}")
    print(f"  → {result['diagnosis']} ({result['severity']})")
    print(f"  → Fix: {result['recommendations'][0]}\n")

---
## Exercise 2: Debug Feature Importance Issues

Identify problematic features in a dataset.

**Difficulty:** Medium | **Skill:** Feature debugging

In [ ]:
def audit_features(df: pd.DataFrame, target_col: str) -> dict:
    """
    Audit features for common issues.
    
    Check for:
    - High missing values
    - Low variance features
    - High correlation with target (leakage)
    - High cardinality categoricals
    
    Returns:
        Dict with problematic features by issue type
    """
    # YOUR CODE HERE
    pass


# Test data
np.random.seed(42)
test_df = pd.DataFrame({
    'good_feature': np.random.randn(100),
    'mostly_missing': [np.nan] * 90 + list(range(10)),
    'constant': [1] * 100,
    'id_column': range(100),
    'target': np.random.randint(0, 2, 100)
})

issues = audit_features(test_df, 'target')
print("Feature Issues Found:")
for issue_type, features in issues.items():
    if features:
        print(f"  {issue_type}: {features}")

In [ ]:
# Solution
def audit_features_solution(df: pd.DataFrame, target_col: str) -> dict:
    issues = {
        'high_missing': [],
        'low_variance': [],
        'potential_leakage': [],
        'high_cardinality': []
    }
    
    feature_cols = [c for c in df.columns if c != target_col]
    
    for col in feature_cols:
        # Check missing values (>50%)
        missing_pct = df[col].isna().mean()
        if missing_pct > 0.5:
            issues['high_missing'].append((col, f"{missing_pct:.1%}"))
        
        # Check low variance (for numeric)
        if df[col].dtype in ['int64', 'float64']:
            non_null = df[col].dropna()
            if len(non_null) > 0:
                unique_ratio = non_null.nunique() / len(non_null)
                if unique_ratio < 0.01:  # Less than 1% unique
                    issues['low_variance'].append((col, f"{non_null.nunique()} unique"))
        
        # Check for potential leakage (very high correlation)
        if df[col].dtype in ['int64', 'float64']:
            corr = df[[col, target_col]].corr().iloc[0, 1]
            if abs(corr) > 0.95:
                issues['potential_leakage'].append((col, f"corr={corr:.3f}"))
        
        # Check high cardinality (>50% unique for object type)
        if df[col].dtype == 'object':
            unique_ratio = df[col].nunique() / len(df)
            if unique_ratio > 0.5:
                issues['high_cardinality'].append((col, f"{df[col].nunique()} unique"))
        
        # Also check numeric columns that look like IDs
        if df[col].dtype in ['int64', 'float64']:
            if df[col].nunique() == len(df):
                issues['high_cardinality'].append((col, "unique per row (ID?)"))
    
    return issues

issues = audit_features_solution(test_df, 'target')
print("Feature Issues Found:")
for issue_type, features in issues.items():
    if features:
        print(f"\n  {issue_type.upper()}:")
        for feat, detail in features:
            print(f"    - {feat}: {detail}")

---
## Exercise 3: Learning Curve Analysis

Generate and interpret learning curves to diagnose issues.

**Difficulty:** Medium | **Skill:** Model diagnostics

In [ ]:
def analyze_learning_curve(model, X, y, cv: int = 5) -> dict:
    """
    Generate and analyze learning curves.
    
    Returns:
        Dict with:
        - train_sizes: Array of training sizes used
        - train_scores: Mean train scores at each size
        - val_scores: Mean validation scores at each size
        - diagnosis: What the curves suggest
    """
    # YOUR CODE HERE
    pass


# Test
X, y = make_classification(n_samples=500, n_features=20, random_state=42)

# Overfitting model (deep tree)
result = analyze_learning_curve(DecisionTreeClassifier(max_depth=None), X, y)
print("Deep Tree Diagnosis:", result['diagnosis'])

In [ ]:
# Solution
from sklearn.model_selection import learning_curve

def analyze_learning_curve_solution(model, X, y, cv: int = 5) -> dict:
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy'
    )
    
    train_mean = train_scores.mean(axis=1)
    val_mean = val_scores.mean(axis=1)
    
    # Analyze the curves
    final_train = train_mean[-1]
    final_val = val_mean[-1]
    gap = final_train - final_val
    
    # Check if val score is still improving
    val_improvement = val_mean[-1] - val_mean[-3] if len(val_mean) >= 3 else 0
    
    if gap > 0.15:
        if val_improvement > 0.02:
            diagnosis = "OVERFITTING - More data may help"
        else:
            diagnosis = "OVERFITTING - Reduce model complexity"
    elif final_val < 0.7:
        diagnosis = "UNDERFITTING - Increase model complexity"
    elif gap < 0.05 and final_val > 0.8:
        diagnosis = "GOOD FIT"
    else:
        diagnosis = "MODERATE FIT - Room for improvement"
    
    return {
        'train_sizes': train_sizes,
        'train_scores': train_mean,
        'val_scores': val_mean,
        'gap': gap,
        'diagnosis': diagnosis
    }

# Test with different models
models = [
    ('Deep Tree', DecisionTreeClassifier(max_depth=None)),
    ('Shallow Tree', DecisionTreeClassifier(max_depth=3)),
    ('Random Forest', RandomForestClassifier(n_estimators=50))
]

for name, model in models:
    result = analyze_learning_curve_solution(model, X, y)
    print(f"{name}:")
    print(f"  Train: {result['train_scores'][-1]:.3f}, Val: {result['val_scores'][-1]:.3f}")
    print(f"  Gap: {result['gap']:.3f}")
    print(f"  Diagnosis: {result['diagnosis']}\n")

---
## Exercise 4: Debug a Broken Pipeline

Find and fix bugs in this ML pipeline.

**Difficulty:** Hard | **Skill:** Debugging

In [ ]:
# BUGGY PIPELINE - Find and list all the bugs!

def buggy_pipeline(df, target_col):
    """This pipeline has at least 5 bugs. Find them all!"""
    
    # Bug 1: Something wrong with this split
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99)
    
    # Bug 2: Issue with scaling
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.fit_transform(X_test)  # Bug here!
    
    # Bug 3: Model issue
    model = LogisticRegression(max_iter=1)  # Bug here!
    model.fit(X_train_scaled, y_train)
    
    # Bug 4: Evaluation issue
    train_pred = model.predict(X_train_scaled)
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_train, train_pred)  # Bug here!
    
    # Bug 5: Return issue
    return train_acc  # Bug: Missing test_acc


# YOUR TASK: List all bugs and write the corrected version below
def list_bugs():
    """Return a list of all bugs found."""
    bugs = [
        # "Bug 1: ...",
        # "Bug 2: ...",
        # etc.
    ]
    return bugs

print("Bugs found:", len(list_bugs()))

In [ ]:
# Solution
def list_bugs_solution():
    return [
        "Bug 1: test_size=0.99 leaves only 1% for training (should be ~0.2-0.3)",
        "Bug 2: Calling fit_transform on test set causes data leakage (should be transform only)",
        "Bug 3: max_iter=1 is too low, model won't converge (should be 100+)",
        "Bug 4: Evaluating test accuracy using y_train and train_pred instead of y_test and test_pred",
        "Bug 5: Only returning train_acc, missing test_acc and model"
    ]


def fixed_pipeline(df, target_col):
    """Corrected pipeline."""
    from sklearn.preprocessing import StandardScaler
    
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    
    # Fix 1: Proper test size
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    # Fix 2: Only transform test set
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)  # Fixed!
    
    # Fix 3: Proper max_iter
    model = LogisticRegression(max_iter=1000)  # Fixed!
    model.fit(X_train_scaled, y_train)
    
    # Fix 4: Correct evaluation
    train_pred = model.predict(X_train_scaled)
    test_pred = model.predict(X_test_scaled)  # Fixed!
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)  # Fixed!
    
    # Fix 5: Return everything useful
    return {
        'model': model,
        'scaler': scaler,
        'train_acc': train_acc,
        'test_acc': test_acc
    }

# Print bugs
print("All bugs found:")
for i, bug in enumerate(list_bugs_solution(), 1):
    print(f"  {i}. {bug}")

# Test fixed pipeline
X, y = make_classification(n_samples=200, n_features=10, random_state=42)
test_df = pd.DataFrame(X, columns=[f'f{i}' for i in range(10)])
test_df['target'] = y

result = fixed_pipeline(test_df, 'target')
print(f"\nFixed Pipeline Results:")
print(f"  Train Accuracy: {result['train_acc']:.3f}")
print(f"  Test Accuracy: {result['test_acc']:.3f}")

---
## Exercise 5: Explain Model Behavior

Given a scenario, explain what's happening and how to fix it.

**Difficulty:** Medium | **Skill:** ML reasoning

In [ ]:
# Scenario-based debugging
scenarios = {
    "scenario_1": {
        "description": "Model has 99% accuracy on both train and test, but fails in production",
        "possible_causes": [],  # YOUR ANSWER
        "debugging_steps": []   # YOUR ANSWER
    },
    "scenario_2": {
        "description": "Cross-validation gives 85% accuracy, but holdout test gives 60%",
        "possible_causes": [],
        "debugging_steps": []
    },
    "scenario_3": {
        "description": "Model accuracy drops from 90% to 70% after a month in production",
        "possible_causes": [],
        "debugging_steps": []
    }
}

# Fill in the answers for each scenario

In [ ]:
# Solution
scenarios_solution = {
    "scenario_1": {
        "description": "Model has 99% accuracy on both train and test, but fails in production",
        "possible_causes": [
            "Data distribution shift between training and production",
            "Data leakage - test set is too similar to training",
            "Different preprocessing in production",
            "Feature engineering differences",
            "Label definition changed"
        ],
        "debugging_steps": [
            "Compare feature distributions between train and production data",
            "Check for data leakage in feature engineering",
            "Verify preprocessing pipeline is identical",
            "Collect and label production samples, evaluate model",
            "Check for temporal patterns (time-based split needed?)"
        ]
    },
    "scenario_2": {
        "description": "Cross-validation gives 85% accuracy, but holdout test gives 60%",
        "possible_causes": [
            "Holdout set is from different time period (temporal shift)",
            "Holdout set has different distribution",
            "Overfitting during hyperparameter tuning",
            "Data leakage in CV but not in holdout",
            "Stratification issues"
        ],
        "debugging_steps": [
            "Check if holdout is temporally separated",
            "Compare feature distributions between CV folds and holdout",
            "Use nested CV for hyperparameter tuning",
            "Check class balance in holdout vs training",
            "Review feature engineering for any leakage"
        ]
    },
    "scenario_3": {
        "description": "Model accuracy drops from 90% to 70% after a month in production",
        "possible_causes": [
            "Concept drift - underlying patterns changed",
            "Data drift - input distribution changed",
            "Upstream data pipeline changes",
            "Seasonality effects",
            "Feature became unavailable or corrupted"
        ],
        "debugging_steps": [
            "Monitor feature distributions over time",
            "Implement drift detection metrics",
            "Check for missing or corrupted features",
            "Retrain on recent data and compare",
            "Analyze errors by time period"
        ]
    }
}

for name, scenario in scenarios_solution.items():
    print(f"\n{'='*60}")
    print(f"SCENARIO: {scenario['description']}")
    print(f"{'='*60}")
    print("\nPossible Causes:")
    for i, cause in enumerate(scenario['possible_causes'], 1):
        print(f"  {i}. {cause}")
    print("\nDebugging Steps:")
    for i, step in enumerate(scenario['debugging_steps'], 1):
        print(f"  {i}. {step}")

---
## 📋 Week 6 Practice Summary

**Exercises Completed:**
- [ ] Diagnose Overfitting vs Underfitting
- [ ] Debug Feature Importance Issues
- [ ] Learning Curve Analysis
- [ ] Debug Broken Pipeline
- [ ] Explain Model Behavior Scenarios

**Key Skills:**
- Bias-variance tradeoff diagnosis
- Feature auditing
- Learning curve interpretation
- Pipeline debugging
- Production failure analysis

---
**Ready for Week 7: Coding & Communication!** 🚀